In [1]:
import os
from io import BytesIO

from dotenv import load_dotenv
import boto3
import pandas as pd
from google.cloud import bigquery
import sys
import duckdb
from pathlib import Path

module_dir = Path('/home/gaolgo/documents/repositories/murta_engenharia/relatorio_geral/etl_20')
sys.path.insert(0, str(module_dir))
from  bradesco_etl_silver import query_df_from_bigquery

load_dotenv("/home/gaolgo/documents/repositories/murta_engenharia/.env")

storage_options = {
    "key": os.getenv("AWS_ACCESS_KEY_ID"),
    "secret": os.getenv("AWS_SECRET_ACCESS_KEY")}

## APP Murta & Murta Cloud

In [12]:
anomes = "0326"
mes = int(anomes[:2])
ano = 2000 + int(anomes[2:])

inicio = pd.Timestamp(year=ano, month=mes, day=1)
inicio_str = inicio.strftime("%Y-%m-%d %H:%M:%S")

# APP MURTA
relatorio_app_murta = pd.read_parquet(
    "s3://murta-ia-iden-data/iden_app_murta_data.parquet",
    storage_options=storage_options,
    engine="pyarrow",)

relatorio_app_murta["Data de Internação"] = pd.to_datetime(
    relatorio_app_murta["Data de Internação"],
    format="%d/%m/%Y",
    errors="coerce",)

relatorio_app_murta = relatorio_app_murta[
    relatorio_app_murta["Data de Internação"] >= inicio]

# MURTA CLOUD
con = duckdb.connect()
con.execute("LOAD httpfs")
con.execute(f"SET s3_access_key_id='{storage_options['key']}'")
con.execute(f"SET s3_secret_access_key='{storage_options['secret']}'")

relatorio_murta_cloud_df = con.execute("""
    SELECT *
    FROM read_parquet('s3://murta-ia-iden-data/iden_murta_cloud_data.parquet')
    WHERE "data_de_internacao_app_murta" >= ?
""", [inicio_str]).fetch_df()

## Bradesco

In [ ]:
caminho_sql = Path("sql_bradesco.txt")
bradesco_anomes = "1225"
arquivo_excel = f"relatorio_bradesco_geral_{bradesco_anomes}.xlsx"

sql_geral = Path("sql_bradesco.txt").read_text(encoding="utf-8")

client = bigquery.Client(project="power-bi-data-455019")

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("anomes", "STRING", bradesco_anomes)])

df_geral = client.query(sql_geral, job_config=job_config).to_dataframe()
#df_geral.to_excel(arquivo_excel, index=False)

/home/gaolgo/documents/repositories/murta_engenharia/murta_env/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
